# ClickHouse IMDB database exploration

Exploratory SQL over the `imdb` database (the ClickHouse docs' [IMDB example dataset](https://clickhouse.com/docs/getting-started/example-datasets/imdb)), ported from the earlier Neo4j/Cypher exploration notebook. Graph patterns are expressed as JOINs over these tables:

```
movies(id UInt32, name String, year UInt32, rank Float32)        -- ~388k rows, rank 0 = unrated
actors(id UInt32, first_name String, last_name String, gender FixedString(1))
directors(id UInt32, first_name String, last_name String)
roles(actor_id UInt32, movie_id UInt32, role String, created_at DateTime)   -- ~3.4M rows
movie_directors(director_id UInt32, movie_id UInt64)
genres(movie_id UInt32, genre String)
```

Dataset quirks worth knowing:

- Titles move a leading article to the end: `Matrix, The`, `Few Good Men, A`.
- Movies span 1888-2008; only ~67k of them have a `rank` (IMDb rating) - `rank = 0` means unrated.
- `first_name` may carry a disambiguation suffix (`Chris (I)`); the display name is
  `concat(splitByString(' (', first_name)[1], ' ', last_name)`.
- There is no popularity column. Cast size (`count()` of `roles` per movie) is a good proxy and keeps
  obscure one-actor films with a 9.8 rank out of "top rated" lists.

Create the tables and load the data first with `python clickhouse/load_imdb.py` (the server pulls the dataset from S3). Connection settings come from the `CLICKHOUSE_*` environment variables (see `clickhouse_connection.ipynb`).

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if present; never hard-code credentials here

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_SECURE = os.getenv("CLICKHOUSE_SECURE", "false").strip().lower() in ("1", "true", "yes")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT") or (8443 if CLICKHOUSE_SECURE else 8123))
CLICKHOUSE_USERNAME = os.getenv("CLICKHOUSE_USERNAME", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "imdb")

import clickhouse_connect

# HTTP interface: 8123 (plain) / 8443 (TLS, e.g. ClickHouse Cloud)
client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USERNAME,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
    secure=CLICKHOUSE_SECURE,
)

assert client.ping(), "ClickHouse server is not reachable"
print("Connected to ClickHouse", client.server_version, "| database:", client.database)

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)

def run(sql, parameters=None):
    """Run a SELECT and return the result as a pandas DataFrame."""
    return client.query_df(sql, parameters=parameters)

## Check datatypes

Replaces `CALL apoc.meta.nodeTypeProperties()`: list tables, their row counts and column types.

In [ ]:
client.query("SHOW TABLES").result_rows

In [ ]:
pd.DataFrame(
    [(table, client.command(f"SELECT count() FROM {table}")) for table in ("movies", "actors", "directors", "roles", "movie_directors", "genres")],
    columns=["table", "rows"],
)

In [ ]:
run("DESCRIBE TABLE movies")

In [ ]:
run("""
SELECT table, name AS column, type, position
FROM system.columns
WHERE database = currentDatabase()
ORDER BY table, position
""")

## Top movies

The Neo4j version used `imdbRating`; here the score is `movies.rank` (`0` = unrated). Ranking on `rank` alone surfaces obscure films with one credited actor and a 9.8 rating, so we require a cast size of at least 30 (the number of `roles` rows per movie) as a popularity proxy.

In [ ]:
run("""
SELECT m.name AS title, m.year AS year, round(m.rank, 1) AS rank, c.cast_size AS cast_size
FROM movies AS m
INNER JOIN (
    SELECT movie_id, count() AS cast_size
    FROM roles
    GROUP BY movie_id
    HAVING cast_size >= {min_cast:UInt32}
) AS c ON c.movie_id = m.id
WHERE m.rank > 0
ORDER BY m.rank DESC, c.cast_size DESC
LIMIT 10
""", parameters={"min_cast": 30})

## Movies per genre

In [ ]:
run("""
SELECT genre AS Genre, count() AS Movies
FROM genres
GROUP BY genre
ORDER BY Movies DESC
""")

## Movies released per decade

The dataset covers 1888-2008, so a per-decade view is more readable than per-year.

In [ ]:
run("""
SELECT intDiv(year, 10) * 10 AS Decade, count() AS Movies
FROM movies
GROUP BY Decade
ORDER BY Decade ASC
""")

## Most active actors

Aggregate `roles` first, then join only the top 10 actor ids to `actors` for their display names.

In [ ]:
run("""
SELECT concat(splitByString(' (', a.first_name)[1], ' ', a.last_name) AS Actor, r.movies AS Movies
FROM (
    SELECT actor_id, uniqExact(movie_id) AS movies
    FROM roles
    GROUP BY actor_id
    ORDER BY movies DESC
    LIMIT 10
) AS r
INNER JOIN actors AS a ON a.id = r.actor_id
ORDER BY Movies DESC
""")

## Most active directors

In [ ]:
run("""
SELECT concat(splitByString(' (', d.first_name)[1], ' ', d.last_name) AS Director, md.movies AS Movies
FROM (
    SELECT director_id, uniqExact(movie_id) AS movies
    FROM movie_directors
    GROUP BY director_id
    ORDER BY movies DESC
    LIMIT 10
) AS md
INNER JOIN directors AS d ON d.id = md.director_id
ORDER BY Movies DESC
""")

## Director filmography

All movies directed by one person; the name is bound server-side with `{director:String}`.

In [ ]:
run("""
SELECT m.name AS title, m.year AS year, round(m.rank, 1) AS rank
FROM movies AS m
WHERE m.id IN (
    SELECT movie_id
    FROM movie_directors
    WHERE director_id IN (
        SELECT d.id
        FROM directors AS d
        WHERE concat(splitByString(' (', d.first_name)[1], ' ', d.last_name) = {director:String}
    )
)
ORDER BY year ASC, title ASC
""", parameters={"director": "Steven Spielberg"})

## Actors and directors

Cypher's `collect()` becomes `groupArray()` / `groupUniqArray()`. To keep the joins cheap the query is restricted to the ten highest-ranked movies of one year (`{year:UInt32}`), and `roles` / `movie_directors` are pre-filtered to those movies before joining.

In [ ]:
run("""
WITH top_movies AS (
    SELECT id FROM movies WHERE year = {year:UInt32} AND rank > 0 ORDER BY rank DESC LIMIT 10
)
SELECT m.name AS title, m.year AS year, groupUniqArray(concat(splitByString(' (', a.first_name)[1], ' ', a.last_name)) AS Actors, groupUniqArray(concat(splitByString(' (', d.first_name)[1], ' ', d.last_name)) AS Directors
FROM movies AS m
INNER JOIN (SELECT movie_id, actor_id FROM roles WHERE movie_id IN (SELECT id FROM top_movies)) AS r ON r.movie_id = m.id
INNER JOIN actors AS a ON a.id = r.actor_id
INNER JOIN (SELECT movie_id, director_id FROM movie_directors WHERE movie_id IN (SELECT id FROM top_movies)) AS md ON md.movie_id = m.id
INNER JOIN directors AS d ON d.id = md.director_id
WHERE m.id IN (SELECT id FROM top_movies)
GROUP BY m.id, m.name, m.year
ORDER BY title
""", parameters={"year": 1994})

## Actor roles in a movie

The cast of *The Godfather* (1972). Remember the IMDB title convention: the stored name is `Godfather, The`. Title and year are bound server-side with `{title:String}` / `{year:UInt32}`.

In [ ]:
run("""
SELECT concat(splitByString(' (', a.first_name)[1], ' ', a.last_name) AS Actor, r.role AS Role
FROM roles AS r
INNER JOIN actors AS a ON a.id = r.actor_id
WHERE r.movie_id IN (
    SELECT id FROM movies WHERE name = {title:String} AND year = {year:UInt32}
)
ORDER BY Actor
""", parameters={"title": "Godfather, The", "year": 1972})

## Users who rated most movies

**Not ported.** The Neo4j dataset had `(:User)-[:RATED]->(:Movie)` relationships; the IMDB dataset has no users or per-user ratings table, so this query has no equivalent. Only the aggregate `movies.rank` exists.

## Rating distribution

Adapted: the original counted per-user `RATED.rating` values. Here we show the distribution of the movie-level `movies.rank` (rounded to one decimal), excluding unrated movies (`rank = 0`).

In [ ]:
run("""
SELECT round(rank, 1) AS Rating, count() AS Movies
FROM movies
WHERE rank > 0
GROUP BY Rating
ORDER BY Rating
""")

## Duplicate movie check

Same title released in the same year more than once (often distinct productions, e.g. TV episodes or remakes).

In [ ]:
run("""
SELECT name AS title, year, count() AS duplicates
FROM movies
GROUP BY name, year
HAVING duplicates > 1
ORDER BY duplicates DESC, title ASC
LIMIT 50
""")

In [ ]:
client.close()